In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/equal-neutral-submit-version/train_equal.csv
/kaggle/input/equal-neutral-submit-version/test_equal.csv


In [2]:
# ELECTRA:
# - Documentation: https://huggingface.co/google/electra-base-discriminator

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import ShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import ElectraTokenizer, ElectraForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List
import re

# I fixed seed as If someone want to rerun my code, the result should be the same or very close
torch.manual_seed(42)
np.random.seed(42)

2024-08-20 11:23:47.211860: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-20 11:23:47.211986: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-20 11:23:47.328197: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
#Deleting URL and turn text into lowercase
def clean_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = text.lower()
    return text

In [4]:
# I will balance dataset based on the minority class
def balance_data(df, samples_per_class = None):
    if samples_per_class is None:
        samples_per_class = df['sentiment_equal'].value_counts().min()
    balance_data = []
    for sentiment in df['sentiment_equal'].unique():
        class_data = df[df['sentiment_equal'] ==sentiment]
        balance_data.append(class_data.sample(samples_per_class, random_state=42))
    
    balance_df = pd.concat(balance_data, ignore_index=True)
    return balance_df

In [5]:
# Computing Metric
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [6]:
def tokenize_function(data: Dict[str, List], tokenizer: ElectraTokenizer, max_length):
    return tokenizer(data['comment'], padding='max_length', truncation=True, max_length=max_length)

def objective(trial: optuna.Trial, dataset: Dataset, trial_number):
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64])
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-1, log=True)
    max_length = trial.suggest_categorical('max_length', [64, 128])
    
    print(f"\nTrial {trial_number}")
    print(f"Hyperparameters:")
    print(f"  Learning rate: {learning_rate}")
    print(f"  Batch size: {batch_size}")
    print(f"  Weight decay: {weight_decay}")
    print(f"  Max length: {max_length}")
    
    n_splits = 2
    test_size = 0.2
    fold_scores = []
    
    ss = ShuffleSplit(n_splits=n_splits, test_size=test_size, random_state=42)
    
    for fold, (train_index, val_index) in enumerate(ss.split(dataset), 1):
        train_fold = dataset.select(train_index)
        val_fold = dataset.select(val_index)
        
        print(f"Fold {fold}:")
        print(f"  Train size: {len(train_fold)}")
        print(f"  Val size: {len(val_fold)}")
        
        
        tokenizer = ElectraTokenizer.from_pretrained('google/electra-base-discriminator')
        tokenized_train = train_fold.map(lambda x: tokenize_function(x, tokenizer, max_length), batched=True)
        tokenized_val = val_fold.map(lambda x: tokenize_function(x, tokenizer, max_length), batched=True)
        
        model = ElectraForSequenceClassification.from_pretrained('google/electra-base-discriminator', num_labels=3)
        
        training_args = TrainingArguments(
            output_dir=f'./results_trial_{trial_number}_fold_{fold}',
            num_train_epochs=2,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            weight_decay=weight_decay,
            learning_rate=learning_rate,
            evaluation_strategy="epoch",
            save_strategy="no",
            load_best_model_at_end=False,
            metric_for_best_model="f1",
            report_to="none"
        )
        
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_train,
            eval_dataset=tokenized_val,
            compute_metrics=compute_metrics,
        )
        
        trainer.train()
        eval_result = trainer.evaluate()
        fold_scores.append(eval_result['eval_f1'])
        print(f"  Fold {fold} F1 score: {eval_result['eval_f1']:.4f}")
    mean_score = np.mean(fold_scores)
    print(f"Mean F1 score across folds: {mean_score:.4f}")
    
    return mean_score

In [7]:
train_df = pd.read_csv('/kaggle/input/equal-neutral-submit-version/train_equal.csv')
test_df = pd.read_csv('/kaggle/input/equal-neutral-submit-version/test_equal.csv')

for df in [train_df, test_df]:
    df['comment'] = df['comment'].apply(clean_text)
sentiment_mapping = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
train_df['label'] = train_df['sentiment_equal'].map(sentiment_mapping)
test_df['label'] = test_df['sentiment_equal'].map(sentiment_mapping)

In [8]:
balance_train_df = balance_data(train_df, samples_per_class=18750)
print(balance_train_df.head())  
print(balance_train_df['sentiment_equal'].value_counts())
balance_test_df = balance_data(test_df)
print(balance_test_df['sentiment_equal'].value_counts())
train_dataset = Dataset.from_pandas(balance_train_df)
test_dataset = Dataset.from_pandas(balance_test_df)

               group_category  \
0        Business and Finance   
1           Technology and IT   
2  Marketing and Productivity   
3        Business and Finance   
4           Technology and IT   

                                             comment       id_x  rate  \
0  i have just gone through some initial videos a...   46049090   5.0   
1  a great course for updated sccm 2016 tutorials...   21845124   4.5   
2  this course helps in every aspects to write yo...     341868   5.0   
3  this course is really very helpful but it woul...  104319484   5.0   
4                 i learn so much from this training   95034662   4.5   

  sentiment_strict sentiment_moderate sentiment_bit_negative sentiment_equal  \
0         Positive           Positive               Positive        Positive   
1         Positive           Positive               Positive        Positive   
2         Positive           Positive               Positive        Positive   
3         Positive           Positive     

In [9]:
study = optuna.create_study(direction='maximize')
study.optimize(lambda trial: objective(trial, train_dataset, trial.number), n_trials=15)

[I 2024-08-20 11:24:16,436] A new study created in memory with name: no-name-4df2ad73-b4b1-4ad3-ab5e-f46457bc4bec



Trial 0
Hyperparameters:
  Learning rate: 0.0002170593625081512
  Batch size: 32
  Weight decay: 1.9797973827415138e-05
  Max length: 128
Fold 1:
  Train size: 45000
  Val size: 11250


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

... (194 outputs left out)

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.561050,0.765689,0.755351,0.762377,0.765689
2,0.564100,0.527774,0.770667,0.767883,0.766797,0.770667


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


[I 2024-08-20 20:04:36,447] Trial 14 finished with value: 0.7689210806340963 and parameters: {'learning_rate': 3.41603604485845e-05, 'batch_size': 64, 'weight_decay': 0.0005343324693819197, 'max_length': 128}. Best is trial 10 with value: 0.7700737791269889.


  Fold 2 F1 score: 0.7679
Mean F1 score across folds: 0.7689


In [10]:
# Check the best hyperparameter set
best_parameters = study.best_params
print(f"\nBest hyperparameters: {best_parameters}")


Best hyperparameters: {'learning_rate': 4.891879415096683e-05, 'batch_size': 64, 'weight_decay': 0.00013112952605018235, 'max_length': 128}


In [11]:
# Train with the best hyperpameter set and validate on the test set
tokenizer = ElectraTokenizer.from_pretrained('google/electra-base-discriminator')
tokenized_train = train_dataset.map(lambda x: tokenize_function(x, tokenizer, best_parameters['max_length']), batched=True)
tokenized_test = test_dataset.map(lambda x: tokenize_function(x, tokenizer, best_parameters['max_length']), batched=True)

model = ElectraForSequenceClassification.from_pretrained('google/electra-base-discriminator', num_labels=3)

training_args = TrainingArguments(
    output_dir='./final_model',
    num_train_epochs=2,
    per_device_train_batch_size=best_parameters['batch_size'],
    per_device_eval_batch_size=best_parameters['batch_size'],
    weight_decay=best_parameters['weight_decay'],
    learning_rate=best_parameters['learning_rate'],
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()

final_metrics = trainer.evaluate()
print("\nFinal Metrics:")
for metric, value in final_metrics.items():
    print(f"{metric}: {value:.4f}")

Map:   0%|          | 0/56250 [00:00<?, ? examples/s]

Map:   0%|          | 0/64134 [00:00<?, ? examples/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at google/electra-base-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.539923,0.768875,0.763930,0.766639,0.768875
2,0.562900,0.526926,0.776967,0.774018,0.772947,0.776967


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '



Final Metrics:
eval_loss: 0.5269
eval_accuracy: 0.7770
eval_f1: 0.7740
eval_precision: 0.7729
eval_recall: 0.7770
eval_runtime: 265.0775
eval_samples_per_second: 241.9440
eval_steps_per_second: 1.8940
epoch: 2.0000
